In [2]:
from jupyter_dash import JupyterDash

import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
JupyterDash.infer_jupyter_proxy_config()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from CRUD_Python_Module import AnimalShelter

import base64
logo_path = "./Grazioso Salvare Logo.png"

with open(logo_path, "rb") as f:
    encoded_logo = base64.b64encode(f.read()).decode()


###########################
# Data Manipulation / Model
###########################

username = "aacuser"
password = "#F-101Voodoo"

db = AnimalShelter(username, password)

df = pd.DataFrame.from_records(db.read({}))
df.drop(columns=['_id'], inplace=True)


#########################
# Query Builder Function
#########################

def build_query(rescue_type):
    if rescue_type == "water":
        return {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "Labrador Retriever Mix",
                    "Chesapeake Bay Retriever",
                    "Newfoundland"
                ]
            },
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }

    elif rescue_type == "mountain":
        return {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "German Shepherd",
                    "Alaskan Malamute",
                    "Old English Sheepdog",
                    "Siberian Husky",
                    "Rottweiler"
                ]
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }

    elif rescue_type == "disaster":
        return {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "Doberman Pinscher",
                    "German Shepherd",
                    "Golden Retriever",
                    "Bloodhound",
                    "Rottweiler"
                ]
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}
        }

    return {}  # reset


#########################
# Dashboard Layout / View
#########################

app = JupyterDash(__name__)

app.layout = html.Div([

    # Logo + Branding
html.Div([
    html.Img(
        src="data:image/png;base64,{}".format(encoded_logo),
        style={'height': '80px', 'margin-right': '20px'}
    ),
    html.A(" Grazioso Salvare", href="https://www.snhu.edu")
], style={'display': 'flex', 'align-items': 'center'}),

    html.Center(html.B(html.H1('CS-340 Dashboard - Alexander Pessinis'))),
    html.Hr(),

    # Filter Controls
    html.Div([
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Water Rescue', 'value': 'water'},
                {'label': 'Mountain/Wilderness Rescue', 'value': 'mountain'},
                {'label': 'Disaster/Individual Tracking', 'value': 'disaster'},
                {'label': 'Reset', 'value': 'reset'}
            ],
            value='reset',
            labelStyle={'display': 'inline-block', 'margin-right': '20px'}
        )
    ]),

    html.Hr(),

    # Data Table
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),
        row_selectable='single',
        selected_rows=[0],
        page_size=10,
        sort_action='native',
        filter_action='native',
        style_table={'overflowX': 'auto'},
        style_cell={'textAlign': 'left'}
    ),

    html.Br(),
    html.Hr(),

    # Charts Row
    html.Div(className='row', style={'display': 'flex'}, children=[
        html.Div(id='graph-id', className='col s12 m6'),
        html.Div(id='map-id', className='col s12 m6')
    ])
])


#############################################
# Interaction Between Components / Controller
#############################################

# Update DataTable based on filter selection
@app.callback(
    Output('datatable-id', 'data'),
    Input('filter-type', 'value')
)
def update_dashboard(filter_type):
    query = build_query(filter_type)
    results = list(db.read(query))
    dff = pd.DataFrame.from_records(results)
    if '_id' in dff.columns:
        dff.drop(columns=['_id'], inplace=True)
    return dff.to_dict('records')


# Update Pie Chart callback
@app.callback(
    Output('graph-id', "children"),
    Input('datatable-id', "derived_virtual_data")
)
def update_graphs(viewData):
    if viewData is None:
        return []

    dff = pd.DataFrame.from_dict(viewData)
    if dff.empty:
        return []

    counts = dff['breed'].value_counts(normalize=True) * 100
    major = counts[counts >= 5].index.tolist()
    minor = counts[counts < 5].index.tolist()
    dff['breed_grouped'] = dff['breed'].apply(lambda b: b if b in major else "Other")
    fig = px.pie(
        dff,
        names='breed_grouped',
        title='Breed Distribution (Grouped <5% as Other)'
    )

    return [dcc.Graph(figure=fig)]
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    Input('datatable-id', 'selected_columns')
)
def update_styles(selected_columns):
    if not selected_columns:
        return []
    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in selected_columns]

@app.callback(
    Output('map-id', "children"),
    [
        Input('datatable-id', "derived_virtual_data"),
        Input('datatable-id', "derived_virtual_selected_rows")
    ]
)
def update_map(viewData, index):
    if viewData is None or index is None:
        return

    dff = pd.DataFrame.from_dict(viewData)

    if index is None or len(index) == 0:
        row = 0
    else:
        row = index[0]

    return [
        dl.Map(
            style={'width': '1000px', 'height': '500px'},
            center=[30.75, -97.48],
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),
dl.Marker(
    position=[dff.iloc[row]['location_lat'], dff.iloc[row]['location_long']],
    children=[
        dl.Tooltip(dff.iloc[row]['breed']),
        dl.Popup([
            html.H1("Animal Name"),
            html.P(dff.iloc[row]['name'])
        ])
    ]
)
            ]
        )
    ]


# Run App
app.run_server()

Dash app running on https://aprilsubway-symbolbronze-3000.codio.io/proxy/8050/


In [5]:
import os
os.listdir(".")

['ProjectOneTestScript.ipynb',
 'ModuleFiveAssignment.ipynb',
 'CRUD_Python_Module.py',
 'ProjectTwoDashboard.ipynb',
 'ModuleFourTestScript.ipynb',
 'ModuleSixMilestone.ipynb',
 'Tutorial_SampleCode.ipynb',
 '.ipynb_checkpoints',
 '__pycache__',
 'Grazioso Salvare Logo.png']